# Isochrone Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/02-isochrone-analysis.ipynb)

This notebook covers isochrone generation in depth:

- What isochrones are and why they matter
- Travel modes: walk, bike, drive
- Routing backends for performance
- Comparing coverage areas
- Multi-isochrone analysis

## Setup

In [ ]:
# Install SocialMapper with routing support
!pip install -q socialmapper[routing]

In [ ]:
import os
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

from socialmapper import create_isochrone
print("Ready!")

## What is an Isochrone?

An **isochrone** (from Greek: iso = equal, chronos = time) is a polygon that represents all locations reachable from a starting point within a given time limit.

Unlike simple radius-based buffers, isochrones account for:
- Road networks and paths
- Travel mode (walking, biking, driving)
- Terrain and obstacles

In [ ]:
# Create a basic isochrone
isochrone = create_isochrone(
    location="Chicago, IL",
    travel_time=15,
    travel_mode="drive"
)

# Examine the structure
print("Isochrone structure:")
print(f"  Type: {isochrone['type']}")
print(f"  Geometry type: {isochrone['geometry']['type']}")
print(f"\nProperties:")
for key, value in isochrone['properties'].items():
    print(f"  {key}: {value}")

## Travel Modes

SocialMapper supports three travel modes:

| Mode | Description | Typical Speed |
|------|-------------|---------------|
| `walk` | Pedestrian paths | ~5 km/h |
| `bike` | Cycling routes | ~15 km/h |
| `drive` | Road network | ~30-50 km/h |

In [ ]:
# Compare travel modes for the same location and time
location = "San Francisco, CA"
travel_time = 15

modes = ["walk", "bike", "drive"]
results = {}

print(f"15-minute travel from {location}:")
print("=" * 40)

for mode in modes:
    iso = create_isochrone(
        location=location,
        travel_time=travel_time,
        travel_mode=mode
    )
    area = iso['properties']['area_sq_km']
    results[mode] = area
    print(f"{mode.capitalize():8} {area:>8.2f} km²")

# Show relative sizes
print("\nRelative coverage:")
walk_area = results['walk']
print(f"  Biking covers {results['bike']/walk_area:.1f}x walking area")
print(f"  Driving covers {results['drive']/walk_area:.1f}x walking area")

## Routing Backends

SocialMapper supports multiple routing backends:

| Backend | Speed | Requires API Key | Best For |
|---------|-------|------------------|----------|
| `valhalla` | Fast | No | General use (default) |
| `osrm` | Fast | No | Driving only |
| `ors` | Fast | Yes (free) | All modes |
| `networkx` | Slow | No | Offline/precise |

In [ ]:
# Compare backends
import time

location = "Denver, CO"
backends = ["valhalla", "networkx"]

print(f"Backend comparison for {location}:")
print("=" * 50)

for backend in backends:
    try:
        start = time.time()
        iso = create_isochrone(
            location=location,
            travel_time=10,
            backend=backend
        )
        elapsed = time.time() - start
        area = iso['properties']['area_sq_km']
        print(f"{backend:12} Time: {elapsed:.2f}s  Area: {area:.2f} km²")
    except Exception as e:
        print(f"{backend:12} Error: {e}")

## Location Input Methods

You can specify locations as:
1. Place names (geocoded automatically)
2. Coordinates (latitude, longitude)

In [ ]:
# Method 1: Place name
iso1 = create_isochrone(
    location="Empire State Building, New York",
    travel_time=10
)
print(f"Empire State Building: {iso1['properties']['area_sq_km']:.2f} km²")

# Method 2: Coordinates (lat, lon)
iso2 = create_isochrone(
    location=(40.7484, -73.9857),  # Empire State Building coords
    travel_time=10
)
print(f"Same location by coords: {iso2['properties']['area_sq_km']:.2f} km²")

## Multi-Time Analysis

Create isochrones for multiple travel times to show accessibility rings.

In [ ]:
# Create concentric isochrones
location = "Boston, MA"
times = [5, 10, 15, 20, 30]

print(f"Walking accessibility from {location}:")
print("=" * 40)

isochrones = []
for t in times:
    iso = create_isochrone(
        location=location,
        travel_time=t,
        travel_mode="walk"
    )
    isochrones.append(iso)
    area = iso['properties']['area_sq_km']
    print(f"{t:>3} min: {area:>6.2f} km²")

# Show how area grows with time
print("\nArea growth pattern:")
base_area = isochrones[0]['properties']['area_sq_km']
for i, t in enumerate(times):
    area = isochrones[i]['properties']['area_sq_km']
    multiplier = area / base_area
    print(f"  {t} min is {multiplier:.1f}x the 5-min area")

## Practical Example: Site Accessibility Comparison

Compare the accessibility of different potential locations for a new facility.

In [ ]:
# Compare potential sites
sites = [
    ("Downtown Seattle", (47.6062, -122.3321)),
    ("Capitol Hill", (47.6253, -122.3222)),
    ("University District", (47.6597, -122.3134)),
]

print("Site Accessibility Analysis (15-min walk):")
print("=" * 50)

results = []
for name, coords in sites:
    iso = create_isochrone(
        location=coords,
        travel_time=15,
        travel_mode="walk"
    )
    area = iso['properties']['area_sq_km']
    results.append((name, area, iso))
    print(f"{name:25} {area:.2f} km²")

# Find best location
best = max(results, key=lambda x: x[1])
print(f"\nBest pedestrian accessibility: {best[0]} ({best[1]:.2f} km²)")

## Working with Isochrone Geometry

The isochrone result is GeoJSON, which you can use with other geographic tools.

In [ ]:
from shapely.geometry import shape

# Create an isochrone
iso = create_isochrone("Portland, OR", travel_time=15)

# Convert to Shapely geometry for spatial operations
polygon = shape(iso['geometry'])

print("Geometry properties:")
print(f"  Type: {polygon.geom_type}")
print(f"  Valid: {polygon.is_valid}")
print(f"  Area: {polygon.area:.6f} sq degrees")

# Get bounds
bounds = polygon.bounds
print(f"  Bounds: ({bounds[0]:.4f}, {bounds[1]:.4f}) to ({bounds[2]:.4f}, {bounds[3]:.4f})")

# Get centroid
centroid = polygon.centroid
print(f"  Centroid: ({centroid.y:.4f}, {centroid.x:.4f})")

## Checking Point Containment

Test whether specific locations fall within the isochrone.

In [ ]:
from shapely.geometry import shape, Point

# Create isochrone from Space Needle
space_needle = (47.6205, -122.3493)
iso = create_isochrone(space_needle, travel_time=10, travel_mode="walk")
polygon = shape(iso['geometry'])

# Test some locations
test_locations = [
    ("Pike Place Market", (47.6097, -122.3422)),
    ("Seattle Art Museum", (47.6073, -122.3380)),
    ("University of Washington", (47.6553, -122.3035)),
]

print("Locations within 10-min walk from Space Needle:")
print("=" * 50)

for name, (lat, lon) in test_locations:
    point = Point(lon, lat)  # Note: Point takes (lon, lat)
    within = polygon.contains(point)
    status = "Yes" if within else "No"
    print(f"{name:30} {status}")

## Exporting Isochrones

Save isochrones for use in other applications.

In [ ]:
import json

# Create an isochrone
iso = create_isochrone("Austin, TX", travel_time=15)

# Save as GeoJSON
with open("austin_isochrone.geojson", "w") as f:
    json.dump(iso, f, indent=2)

print("Saved austin_isochrone.geojson")
print(f"File can be opened in QGIS, Mapbox, or any GeoJSON viewer")

## Exercise: Transportation Equity Analysis

Compare car vs. transit accessibility to understand transportation equity.

In [ ]:
# Analyze transportation equity
location = "Los Angeles, CA"

# What can you reach in 30 minutes by different modes?
drive_30 = create_isochrone(location, travel_time=30, travel_mode="drive")
walk_30 = create_isochrone(location, travel_time=30, travel_mode="walk")

drive_area = drive_30['properties']['area_sq_km']
walk_area = walk_30['properties']['area_sq_km']

print(f"30-minute accessibility in {location}:")
print("=" * 40)
print(f"By car:     {drive_area:>8.2f} km²")
print(f"On foot:    {walk_area:>8.2f} km²")
print(f"")
print(f"Accessibility gap: {drive_area/walk_area:.0f}x")
print(f"\nPeople without cars can access {(walk_area/drive_area)*100:.1f}%")
print(f"of the area available to drivers.")

## Next Steps

Continue with:

- **[Points of Interest](03-points-of-interest.ipynb)** - Find what's within your isochrones
- **[Census Data](04-census-data.ipynb)** - Analyze who lives in these areas
- **[Mapping](05-mapping-visualization.ipynb)** - Visualize your isochrones